# 01. Rolling Median/MAD 서버 이상 탐지

목표: 외부 패키지 없이 CPU, memory, latency와 error rate의 rolling baseline을 만들고 spike와 level shift를 탐지합니다.

MAD는 median absolute deviation입니다. 평균·표준편차보다 극단값에 덜 흔들리므로 작은 서버의 첫 기준선으로 유용합니다.

In [ ]:
import math
import random
from collections import deque
from statistics import median

random.seed(42)

FEATURES = ("cpu_pct", "memory_pct", "p95_latency_ms", "error_rate")

def generate_server_stream(size=300):
    rows = []
    for t in range(size):
        row = {
            "t": t,
            "cpu_pct": 35 + 8 * math.sin(t / 18) + random.gauss(0, 2.0),
            "memory_pct": 52 + 0.015 * t + random.gauss(0, 0.8),
            "p95_latency_ms": 120 + 15 * math.sin(t / 11) + random.gauss(0, 5.0),
            "error_rate": max(0.0, 0.006 + random.gauss(0, 0.002)),
            "is_anomaly": False,
        }
        # 짧고 강한 장애: CPU, latency, error rate가 함께 상승합니다.
        if 170 <= t < 180:
            row["cpu_pct"] += 50
            row["p95_latency_ms"] += 600
            row["error_rate"] += 0.22
            row["is_anomaly"] = True
        # 점진적 memory leak: 한 feature가 서서히 정상 범위를 벗어납니다.
        if 240 <= t < 260:
            row["memory_pct"] += 1.8 * (t - 239)
            row["is_anomaly"] = True
        rows.append(row)
    return rows

rows = generate_server_stream()
print("rows:", len(rows), "labeled anomalies:", sum(row["is_anomaly"] for row in rows))

## Detector 구현

현재 관측을 먼저 점수화한 뒤 정상으로 판단된 경우에만 학습합니다. anomaly를 곧바로 history에 넣으면 장애가 새로운 정상으로 흡수되는 contamination이 생길 수 있기 때문입니다.

In [ ]:
class RollingMADDetector:
    def __init__(self, features, window_size=60, warmup=40, threshold=6.0):
        self.features = tuple(features)
        self.window_size = window_size
        self.warmup = warmup
        self.threshold = threshold
        self.history = {name: deque(maxlen=window_size) for name in self.features}

    @staticmethod
    def robust_score(value, history):
        center = median(history)
        mad = median(abs(item - center) for item in history)
        # 정규분포에서 MAD/0.6745는 표준편차의 robust 추정치입니다.
        scale = max(mad / 0.6745, 1e-6)
        return abs(value - center) / scale

    def process(self, row):
        warmed = all(len(self.history[name]) >= self.warmup for name in self.features)
        if warmed:
            scores = {name: self.robust_score(row[name], self.history[name]) for name in self.features}
        else:
            scores = {name: 0.0 for name in self.features}
        max_feature = max(scores, key=scores.get)
        is_alert = warmed and scores[max_feature] > self.threshold
        # 학습 초기는 모두 반영하고, 이후 alert 관측은 baseline 오염을 막기 위해 건너뜁니다.
        if not is_alert:
            for name in self.features:
                self.history[name].append(row[name])
        return is_alert, max_feature, scores

detector = RollingMADDetector(FEATURES)
raw_alerts = []
details = []
for row in rows:
    alert, feature, scores = detector.process(row)
    raw_alerts.append(alert)
    details.append((row["t"], alert, feature, scores[feature], row["is_anomaly"]))

print("raw alert points:", sum(raw_alerts))
print("first alerts:")
for item in [item for item in details if item[1]][:8]:
    print(item)

## Persistence gate

한 번의 spike로 paging하지 않고 최근 3개 관측 중 2개 이상이 threshold를 넘을 때만 alert로 확정합니다.

In [ ]:
def persistence_gate(flags, window=3, required=2):
    confirmed = []
    for index in range(len(flags)):
        recent = flags[max(0, index - window + 1):index + 1]
        confirmed.append(sum(recent) >= required)
    return confirmed

confirmed = persistence_gate(raw_alerts)

def classification_metrics(labels, predictions):
    tp = sum(label and pred for label, pred in zip(labels, predictions))
    fp = sum((not label) and pred for label, pred in zip(labels, predictions))
    fn = sum(label and (not pred) for label, pred in zip(labels, predictions))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}

labels = [row["is_anomaly"] for row in rows]
print("raw:", classification_metrics(labels, raw_alerts))
print("persistent:", classification_metrics(labels, confirmed))

합성 데이터의 점수는 모델 비교 근거가 아닙니다. 실제 데이터에서는 시간 순서를 유지하고, 배포·점검 구간을 label로 기록하며, false alerts/day와 mean time to detect를 함께 측정하세요.